# Generate Ground Truth Data

source: https://www.youtube.com/watch?v=KoyYkv8P_jU&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=37

In [1]:
import sys
import os
import json
import pandas as pd
from rich import print

parent_directory = os.path.dirname(os.getcwd())
sys.path.append(parent_directory)
from evaluation_utils import llm_structured, Questions, llm_structured_retry
from ingest import load_faq_data

documents = load_faq_data()

# we will create ground truth data for the llm-zoomcamp  to get a small data set
documents = [doc for doc in documents if doc['course'] == 'llm-zoomcamp']

# check the length
print(f'The number of documents for the llm-zoomcamp is : {len(documents)}')
print(documents[0])

# the dictionary (document) is converted to string
user_prompt = json.dumps(documents[0])
print(f'The user prompt is: {user_prompt}')

The number of documents for the llm-zoomcamp is : 103

{
    'id': '74eb249bbf',
    'course': 'llm-zoomcamp',
    'section': 'General Course-Related Questions',
    'question': 'I just discovered the course. Can I still join?',
    'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still 
accepting submissions.'
}

The user prompt is: {"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", 
"question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a 
certificate, you need to submit your project while we\u2019re still accepting submissions."}

### for each doc
1. we get the doc,
2. we dump the doc (convert it to str) using json.dump(doc),
3. we use llm_structed to get the question for each doc,
4. we create a dictionary with keys: question: list_of_questions and the document:doc_id for each question and we append in a list
4. we return the doc and usage (input and output tokens)

In [ ]:

Question_INSTRUCTION = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

MODEL = "gemma3:4b"

# def generate_ground_truth(doc):
#     user_prompt = json.dumps(doc)
#     quesion_with_metadata = dict()
#     result = list()
#     # apply the llm_structed
#     questions, usage_token = llm_structured(
#         instructions=Question_INSTRUCTION ,
#         user_prompt=user_prompt,
#         output_type=Questions, 
#         model=MODEL
#         )
#     for question in questions:
#         quesion_with_metadata = dict()
#         quesion_with_metadata["question"] = question
#         quesion_with_metadata["document"] = doc['id']
#         result.append(quesion_with_metadata)

#     return result, usage_token

# result, usage_token = generate_ground_truth(documents[0])
# print(f'result = {result}')

In [4]:
# we convert the records to pd dataframe to save them later as csv file

# pd.DataFrame(result)

# Generate Ground Truth for all Documents with retry

In [3]:
# We need to do for each doc and this is taking a lot of time if we do for loop
ground_truth = []
usages = []

# sequential processing
# for doc in tqdm(documents[:5]):
#     records, usage = generate_ground_truth(doc)
#     ground_truth.extend(records)
#     usages.append(usage)

In [ ]:
import time
MAX_RETRIES=3
# # note we can also output the time for each call
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)
    quesion_with_metadata = dict()
    result = list()
    questions = list()
    # apply the llm_structed
    questions, usage_token = llm_structured_retry(
        instructions=Question_INSTRUCTION ,
        user_prompt=user_prompt,
        output_type=Questions, 
        model=MODEL,
        max_retries=MAX_RETRIES,
        )
    if questions:
        for question in questions:
            quesion_with_metadata = dict()
            quesion_with_metadata["question"] = question
            quesion_with_metadata["document"] = doc['id']
            result.append(quesion_with_metadata)
        return result, usage_token
    return None, None

result, usage_token = generate_ground_truth(documents[0])
result


[{'question': 'Hi! I’m really interested in this LLM Zoomcamp. I just found it now – is there any chance I can still sign up, even though the course has already started?',
  'document': '74eb249bbf'},
 {'question': 'I saw that you need a project to get a certificate. What exactly does ‘while we’re still accepting submissions’ mean? Is there a deadline for when I have to finish and submit my project?',
  'document': '74eb249bbf'},
 {'question': "Okay, so if I *do* join now, will I be able to get a certificate? It says I only get one if I submit a project during the time you're accepting them – can you explain that a little more?",
  'document': '74eb249bbf'},
 {'question': 'Just wondering - if I’m not planning on getting a certificate, can I still participate in the Zoomcamp and complete the course materials?',
  'document': '74eb249bbf'},
 {'question': 'I read that to get a certificate, I need to submit my project while submissions are accepted. What kind of projects are you looking fo

In [ ]:
pd.DataFrame(result)

,question,document
0,Hi! I’m really interested in this LLM Zoomcamp...,74eb249bbf
1,I saw that you need a project to get a certifi...,74eb249bbf
2,"Okay, so if I *do* join now, will I be able to...",74eb249bbf
3,Just wondering - if I’m not planning on gettin...,74eb249bbf
4,"I read that to get a certificate, I need to su...",74eb249bbf


In [6]:
from tqdm import tqdm  # this is only for monitoring, nothing else

# so in this example I will get how many document will be processed and also the time taken for processing all documents
for doc in tqdm(documents[:4]):
    generate_ground_truth(doc)

100%|██████████| 4/4 [00:23<00:00,  6.00s/it]


In [6]:
from concurrent.futures import ThreadPoolExecutor, as_completed # to run N document at the same time and save cost and time
from tqdm import tqdm # for monitoring

# without tdqm
# # To progress 4 documents in parallel so that we save time
# with ThreadPoolExecutor(max_workers=4) as executor:

#     futures = [
#         executor.submit(generate_ground_truth, doc) # pass function name, and doc
#         for doc in documents
#     ]

#     # for future in as_completed(futures): # this is without tqdm
#     for future in tqdm(as_completed(futures), total=len(futures)):
#         questions, usage = future.result()




# Process all documents in parallel

In [ ]:
# MAX_WORKERS = 4   # Try 2 or 4 for Ollama

# all_questions = []
# all_usage = []

# with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

#     # Start one task for every document
#     futures = [
#         executor.submit(generate_ground_truth, doc)
#         for doc in documents
#     ]

#     # Show progress as tasks complete
#     for future in tqdm(as_completed(futures), total=len(futures), desc="Generating questions"):

#         try:
#             questions, usage = future.result()
#             # Skip failed documents
#             if questions is None:
#                 continue

#             # Add all generated questions
#             all_questions.extend(questions)

#             # Store usage for this document
#             all_usage.append(usage)

#         except Exception as e:
#             # This normally shouldn't happen if llm_structured_retry()
#             # returns (None, None) after retries, but it's good practice.
#             print(f"Unexpected error: {e}")

# print(f"Generated {len(all_questions)} questions")
# print(f"Processed {len(all_usage)} documents successfully")

Generating questions:  69%|██████▉   | 71/103 [06:31<02:56,  5.52s/it]


# Run in parallel all the documents 

In [9]:
# create a function
from evaluation_utils import run_in_parallel, generate_ground_truth

results = run_in_parallel(
    func=generate_ground_truth,
    items=documents,
)

Processing: 100%|██████████| 103/103 [09:07<00:00,  5.32s/it]


In [10]:
all_questions = list()
all_usage = list()

for questions, usage in results:
    if questions is None:
        continue
    all_questions.extend(questions)
    all_usage.append(usage)

In [11]:
len(all_questions)

510

In [17]:
len(documents)

103

In [14]:
all_questions[0]

{'question': "Okay, so I’m really confused about Office Hours. Where do I actually find the Zoom link? It's not publicly available, right?",
 'document': '489dd1c9d9'}

In [15]:
all_usage[0]

{'input_tokens': 307, 'output_tokens': 204, 'total_tokens': 511}

In [16]:
103 * 5

515

In [22]:
total_input_tokens = sum(
    usage["input_tokens"]
    for usage in all_usage
)

total_output_tokens = sum(
    usage["output_tokens"]
    for usage in all_usage
)

total_tokens = sum(
    usage["total_tokens"]
    for usage in all_usage
)

print(f"Total input tokens: {total_input_tokens}")
print(f"Total output tokens: {total_output_tokens}")
print(f"Total tokens: {total_tokens}")

Total input tokens: 32378

Total output tokens: 22379

Total tokens: 54757

# why I get 510 instead of 515 results because some documents have less than 5 questions

In [18]:
from collections import Counter

question_count = Counter(
    q["document"]
    for q in all_questions
)

for doc_id, count in question_count.items():
    if count < 5:
        print(doc_id, count)

04440cab11 2

4b65d5542d 1

4b30b918bc 4

In [19]:
df_ground_truth = pd.DataFrame(all_questions)
df_ground_truth 

,question,document
0,"Okay, so I’m really confused about Office Hour...",489dd1c9d9
1,I saw something about YouTube Live and Slido f...,489dd1c9d9
2,The announcement channel on Telegram and Slack...,489dd1c9d9
3,I noticed you mentioned watching live on DataT...,489dd1c9d9
4,Just to clarify – I shouldn’t be asking questi...,489dd1c9d9
...,...,...
505,"I’ve double-checked my API key and endpoint, b...",0dfba07ace
506,I’m getting a 401 Client Error – what does tha...,4b30b918bc
507,The `requests` library is installing version 2...,4b30b918bc
508,Is there a specific key I need to grant access...,4b30b918bc


In [ ]:
df_ground_truth.to_csv("data/ground_truth_FAQ_data.csv", index=False)